# AgentRL Systems Demo (SFT + Eval + A/B/C)

This notebook runs:
1. SFT bootstrap from scratch
2. Quick bootstrap eval + parse diagnostics
3. Systems A/B/C comparison on GRPO runtime modes

Conditions:
- A: standard
- B: continuous batching
- C: continuous batching + prefix cache

This version is tuned for A100 while remaining Colab-friendly.

In [ ]:
# Cell 1: Install dependencies (torchao-safe)
!pip uninstall -y torchao
!pip install -q -U pip setuptools wheel
!pip install -q -e .
!pip install -q transformers peft accelerate datasets pandas matplotlib seaborn

In [ ]:
# Cell 2: Imports and global config
import re, json, random, gc, shutil
from pathlib import Path

import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from transformers import AutoTokenizer
from peft import LoraConfig

from agentrl import (
    GRPOConfig,
    GRPOTrainer,
    SFTBootstrapTrainer,
    BaseEnvironment,
    BaseVerifier,
)
from agentrl.memory.layout import SharedWeightLayout

SEED = 42
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = "bfloat16" if DEVICE == "cuda" else "float32"
print("DEVICE:", DEVICE, "| DTYPE:", DTYPE)

# Optional: set to True in Colab to persist artifacts.
SAVE_TO_DRIVE = True
DRIVE_ARTIFACT_DIR = Path("/content/drive/MyDrive/agentrl_artifacts")

In [ ]:
# Cell 3: Load GSM8K and create splits
gsm8k = load_dataset("gsm8k", "main")
train_all = gsm8k["train"].shuffle(seed=SEED)

# A100-friendly split sizes for stronger bootstrap and more stable RL signal.
sft_set = train_all.select(range(0, 4000))
rl_train_set = train_all.select(range(4000, 5500))
rl_val_set = train_all.select(range(5500, 6000))

print("SFT:", len(sft_set), "RL train:", len(rl_train_set), "RL val:", len(rl_val_set))

In [ ]:
# Cell 4: Parsing helpers and dataset conversion
ANS_RE = re.compile(r"####\s*([-+]?\d[\d,]*(?:\.\d+)?)")
FINAL_RE = re.compile(r"(?:the answer is|answer:)\s*([-+]?\d[\d,]*(?:\.\d+)?)", re.I)

def normalize_num(s: str) -> str:
    return s.replace(",", "").strip()

def parse_gsm8k_answer(raw_answer: str):
    m = ANS_RE.search(raw_answer)
    final = normalize_num(m.group(1)) if m else None
    reasoning = ANS_RE.sub("", raw_answer).strip()
    return reasoning, final

def extract_final_answer(text: str):
    m = FINAL_RE.search(text)
    if m:
        return normalize_num(m.group(1))
    m2 = re.search(r"([-+]?\d[\d,]*(?:\.\d+)?)\s*$", text.strip())
    return normalize_num(m2.group(1)) if m2 else None

def make_prompt(q: str):
    return (
        "Solve the following math problem.\n"
        "Show concise reasoning and end exactly with: The answer is <number>\n\n"
        f"Question: {q}\n"
    )

def to_records(ds):
    out = []
    for ex in ds:
        reasoning, final = parse_gsm8k_answer(ex["answer"])
        if final is not None:
            out.append({"question": ex["question"], "reasoning": reasoning, "final_answer": final})
    return out

sft_records = to_records(sft_set)
rl_train_records = to_records(rl_train_set)
rl_val_records = to_records(rl_val_set)

print(len(sft_records), len(rl_train_records), len(rl_val_records))

In [ ]:
# Cell 5: Environment and verifier
class GSM8KEnvironment(BaseEnvironment):
    def __init__(self, records, seed=0):
        self.records = list(records)
        self.rng = random.Random(seed)
        self.current = None
        self.last_response = ""

    def reset(self):
        self.current = self.rng.choice(self.records)
        self.last_response = ""
        return make_prompt(self.current["question"])

    def step(self, action):
        self.last_response = action
        return "episode complete", True

    def state(self):
        return {
            "reference_answer": self.current["final_answer"],
            "response": self.last_response,
        }

class GSM8KVerifier(BaseVerifier):
    """Shaped reward: format + numeric closeness + exact match.

    Sparse 0/1 rewards collapse reward_std within GRPO groups. This shaped
    verifier provides intermediate signal so advantages stay informative even
    before the model starts solving problems exactly.
    """

    def _to_float(self, value):
        try:
            return float(str(value).replace(",", "").strip())
        except (TypeError, ValueError):
            return None

    def verify(self, response: str, env_state: dict) -> float:
        pred_raw = extract_final_answer(response)
        gold_raw = env_state["reference_answer"]

        if pred_raw is None:
            return 0.0

        has_final_marker = bool(FINAL_RE.search(response))
        format_reward = 0.1 if has_final_marker else 0.05

        if pred_raw == gold_raw:
            return 1.0

        pred = self._to_float(pred_raw)
        gold = self._to_float(gold_raw)
        if pred is None or gold is None:
            return format_reward

        closeness = max(0.0, 1.0 - abs(pred - gold) / (abs(gold) + 1.0))
        closeness_reward = 0.6 * closeness

        return min(0.95, format_reward + closeness_reward)

def build_sft_samples(records):
    pairs = []
    for r in records:
        prompt = make_prompt(r["question"])
        target = f"{r['reasoning']}\n\nThe answer is {r['final_answer']}"
        pairs.append((prompt, target))
    return pairs

sft_samples = build_sft_samples(sft_records)
print("SFT samples:", len(sft_samples))

In [ ]:
# Cell 6: SFT bootstrap from scratch
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

sft_cfg = GRPOConfig(
    model_name=MODEL_NAME,
    batch_size=4,
    group_size=2,
    max_new_tokens=64,
    steps=1,
    lr=1e-5,
    dtype=DTYPE,
    max_prompt_tokens=768,
    pad_to_multiple_of=8,
    output_dir="./runs/sft_bootstrap",
)

layout = SharedWeightLayout(
    model_name=MODEL_NAME,
    lora_config=lora_cfg,
    dtype=sft_cfg.dtype,
    device=DEVICE,
)

sft_trainer = SFTBootstrapTrainer(config=sft_cfg, tokenizer=tokenizer, layout=layout)
_ = sft_trainer.train(samples=sft_samples, epochs=2, shuffle=True)
sft_adapter_path = str(sft_trainer.save_adapter("./checkpoints/sft_bootstrap"))
print("Saved adapter:", sft_adapter_path)

if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
        drive_adapter_dir = DRIVE_ARTIFACT_DIR / "sft_bootstrap"
        if drive_adapter_dir.exists():
            shutil.rmtree(drive_adapter_dir)
        shutil.copytree(sft_adapter_path, drive_adapter_dir)
        print("Copied SFT adapter to:", drive_adapter_dir)
    except Exception as exc:
        print("Drive copy skipped/failed:", exc)

In [ ]:
# Cell 7: Quick bootstrap eval + diagnostics
@torch.no_grad()
def generate_one(model, tok, prompt, max_new_tokens=128):
    # Ensure greedy decode config does not carry stale sampling knobs.
    if hasattr(model, "generation_config"):
        model.generation_config.do_sample = False
        model.generation_config.temperature = None
        model.generation_config.top_p = None
        model.generation_config.top_k = None

    enc = tok(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tok.pad_token_id,
        eos_token_id=tok.eos_token_id,
    )
    return tok.decode(out[0][enc["input_ids"].shape[-1]:], skip_special_tokens=True)

def eval_bootstrap(adapter_path, records, n=200):
    eval_layout = SharedWeightLayout(
        model_name=MODEL_NAME,
        lora_config=lora_cfg,
        dtype=DTYPE,
        device=DEVICE,
        adapter_path=adapter_path,
    )
    model = eval_layout.model
    if hasattr(model, "set_adapter"):
        model.set_adapter("policy")
    model.eval()

    subset = records[:n]
    correct = 0
    parseable = 0
    strict_final_line = 0

    for r in subset:
        pred_text = generate_one(model, tokenizer, make_prompt(r["question"]), max_new_tokens=128)
        pred = extract_final_answer(pred_text)
        parseable += int(pred is not None)
        correct += int(pred == r["final_answer"])

        last_line = pred_text.strip().splitlines()[-1] if pred_text.strip() else ""
        strict_final_line += int(bool(re.match(r"^\s*(?:The answer is|Answer:)\s*[-+]?\d[\d,]*(?:\.\d+)?\s*$", last_line, flags=re.I)))

    denom = max(1, len(subset))
    return {
        "pass_at_1": correct / denom,
        "parseable_rate": parseable / denom,
        "strict_final_line_rate": strict_final_line / denom,
    }

bootstrap_metrics = eval_bootstrap(sft_adapter_path, rl_val_records, n=200)
print("Bootstrap metrics:", bootstrap_metrics)

In [ ]:
# Cell 8: Optional format-repair continuation (no full SFT restart)
RUN_FORMAT_REPAIR = True
FORMAT_REPAIR_SAMPLES = 800
FORMAT_REPAIR_EPOCHS = 1

if RUN_FORMAT_REPAIR:
    def build_format_repair_samples(records):
        pairs = []
        for r in records[:FORMAT_REPAIR_SAMPLES]:
            prompt = make_prompt(r["question"])
            target = f"The answer is {r['final_answer']}"
            pairs.append((prompt, target))
        return pairs

    format_samples = build_format_repair_samples(sft_records)
    print("Format-repair samples:", len(format_samples))

    repair_cfg = GRPOConfig(
        model_name=MODEL_NAME,
        batch_size=8,
        group_size=2,
        max_new_tokens=32,
        steps=1,
        lr=1e-5,
        dtype=DTYPE,
        max_prompt_tokens=384,
        pad_to_multiple_of=8,
        output_dir="./runs/sft_format_repair",
    )

    repair_layout = SharedWeightLayout(
        model_name=MODEL_NAME,
        lora_config=lora_cfg,
        dtype=repair_cfg.dtype,
        device=DEVICE,
        adapter_path=sft_adapter_path,
    )
    repair_trainer = SFTBootstrapTrainer(config=repair_cfg, tokenizer=tokenizer, layout=repair_layout)
    _ = repair_trainer.train(samples=format_samples, epochs=FORMAT_REPAIR_EPOCHS, shuffle=True)

    sft_adapter_path = str(repair_trainer.save_adapter("./checkpoints/sft_bootstrap_format_repair"))
    print("Saved repaired adapter:", sft_adapter_path)

    bootstrap_metrics = eval_bootstrap(sft_adapter_path, rl_val_records, n=200)
    print("Bootstrap metrics after format-repair:", bootstrap_metrics)

    if SAVE_TO_DRIVE:
        try:
            from google.colab import drive
            drive.mount("/content/drive", force_remount=False)
            DRIVE_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
            drive_adapter_dir = DRIVE_ARTIFACT_DIR / "sft_bootstrap_format_repair"
            if drive_adapter_dir.exists():
                shutil.rmtree(drive_adapter_dir)
            shutil.copytree(sft_adapter_path, drive_adapter_dir)
            print("Copied repaired SFT adapter to:", drive_adapter_dir)
        except Exception as exc:
            print("Drive copy skipped/failed:", exc)

    del repair_trainer
    del repair_layout
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

In [ ]:
# Cell 9: Cleanup before systems runs
for name in ["sft_trainer", "layout"]:
    if name in globals():
        del globals()[name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

train_env = GSM8KEnvironment(rl_train_records, seed=SEED)
train_verifier = GSM8KVerifier()
print("ready")

In [ ]:
# Cell 9: Systems config + runner (A/B/C only)
def make_system_cfg(name, steps=80, seed=42):
    flags = {
        "A_standard": dict(use_continuous_batching=False, use_prefix_cache=False, use_cuda_graph_decode=False),
        "B_continuous": dict(use_continuous_batching=True, use_prefix_cache=False, use_cuda_graph_decode=False),
        "C_continuous_prefix": dict(use_continuous_batching=True, use_prefix_cache=True, use_cuda_graph_decode=False),
    }[name]

    return GRPOConfig(
        model_name=MODEL_NAME,
        init_adapter_path=sft_adapter_path,
        beta=0.0,
        group_size=4,
        batch_size=2,
        max_new_tokens=48,
        max_prompt_tokens=384,
        prefill_chunk_size=128,
        max_episode_steps=1,
        steps=steps,
        lr=1e-5,
        seed=seed,
        temperature=0.8,
        top_p=0.95,
        use_continuous_batching=flags["use_continuous_batching"],
        use_prefix_cache=flags["use_prefix_cache"],
        use_cuda_graph_decode=flags["use_cuda_graph_decode"],
        output_dir=f"./runs/systems_{name}_seed{seed}",
        dtype=DTYPE,
    )

def run_system(name, steps=80, seed=42):
    cfg = make_system_cfg(name, steps=steps, seed=seed)
    trainer = GRPOTrainer(cfg, environment=train_env, verifier=train_verifier)
    _ = trainer.train()
    return cfg.output_dir

In [ ]:
# Cell 10: Run A/B/C separately
out_A = run_system("A_standard", steps=80, seed=42)
print("A done:", out_A)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

out_B = run_system("B_continuous", steps=80, seed=42)
print("B done:", out_B)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

out_C = run_system("C_continuous_prefix", steps=80, seed=42)
print("C done:", out_C)

In [ ]:
# Cell 11: Summarize results
def load_df(run_dir):
    p = Path(run_dir) / "metrics.jsonl"
    return pd.DataFrame([json.loads(line) for line in p.read_text().splitlines()])

dfA = load_df(out_A)
dfB = load_df(out_B)
dfC = load_df(out_C)

summary = pd.DataFrame([
    {"cond":"A", "reward_last5_mean": float(dfA["mean_reward"].tail(5).mean()), "step_time_ms_mean": float(dfA["total_step_time_ms"].mean()), "tokens_per_second_mean": float(dfA["tokens_per_second"].mean()), "peak_vram_mb_max": float(dfA["peak_vram_mb"].max()), "cache_hit_ratio_mean": float(dfA.get("cache_hit_ratio", pd.Series([0])).mean()), "prefill_token_savings_pct_mean": float(dfA.get("prefill_token_savings_pct", pd.Series([0])).mean())},
    {"cond":"B", "reward_last5_mean": float(dfB["mean_reward"].tail(5).mean()), "step_time_ms_mean": float(dfB["total_step_time_ms"].mean()), "tokens_per_second_mean": float(dfB["tokens_per_second"].mean()), "peak_vram_mb_max": float(dfB["peak_vram_mb"].max()), "cache_hit_ratio_mean": float(dfB.get("cache_hit_ratio", pd.Series([0])).mean()), "prefill_token_savings_pct_mean": float(dfB.get("prefill_token_savings_pct", pd.Series([0])).mean())},
    {"cond":"C", "reward_last5_mean": float(dfC["mean_reward"].tail(5).mean()), "step_time_ms_mean": float(dfC["total_step_time_ms"].mean()), "tokens_per_second_mean": float(dfC["tokens_per_second"].mean()), "peak_vram_mb_max": float(dfC["peak_vram_mb"].max()), "cache_hit_ratio_mean": float(dfC.get("cache_hit_ratio", pd.Series([0])).mean()), "prefill_token_savings_pct_mean": float(dfC.get("prefill_token_savings_pct", pd.Series([0])).mean())},
])

print("Bootstrap metrics:", bootstrap_metrics)
print(summary.to_string(index=False))

if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
        run_dirs = {
            "A": Path(out_A) / "checkpoint_final",
            "B": Path(out_B) / "checkpoint_final",
            "C": Path(out_C) / "checkpoint_final",
        }
        for name, src in run_dirs.items():
            dst = DRIVE_ARTIFACT_DIR / f"grpo_{name}_checkpoint_final"
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print(f"Copied {name} adapter to {dst}")
    except Exception as exc:
        print("Drive export skipped/failed:", exc)

sns.set(style="whitegrid")
plt.figure(figsize=(8,4)); sns.barplot(data=summary, x="cond", y="step_time_ms_mean"); plt.title("Mean Step Time"); plt.show()
plt.figure(figsize=(8,4)); sns.barplot(data=summary, x="cond", y="tokens_per_second_mean"); plt.title("Tokens/sec"); plt.show()
plt.figure(figsize=(8,4)); sns.barplot(data=summary, x="cond", y="cache_hit_ratio_mean"); plt.title("Cache Hit Ratio"); plt.show()